In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
def reduce_mem_usage(df):
    """Reduce dataframe memory usage"""
    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()

            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.float32)

    return df
def weighted_rmse_score(y_target, y_pred, w):
    """Competition metric"""
    y_target = np.array(y_target, dtype=np.float64)
    y_pred = np.array(y_pred, dtype=np.float64)
    w = np.array(w, dtype=np.float64)
    
    denom = np.sum(w * y_target ** 2)
    if denom == 0 or np.isnan(denom):
        return 0.0
    
    numerator = np.sum(w * (y_target - y_pred) ** 2)
    ratio = numerator / denom
    clipped = np.clip(ratio, 0.0, 1.0)
    score = np.sqrt(1.0 - clipped)
    
    return float(score)
train_df= reduce_mem_usage(pd.read_parquet('train.parquet'))
test_df = reduce_mem_usage(pd.read_parquet('test.parquet'))

missing_cols = [col for col in train_df.columns if train_df[col].isnull().any() and col not in ['id', 'code', 'sub_code', 'sub_category']]
for col in missing_cols: 
    median_val = train_df[col].median()
    train_df[col] = train_df[col].fillna(median_val)
    # Use train median for test to avoid leakage
    if col in test_df.columns:
        test_df[col] = test_df[col].fillna(median_val)

import sys
sys.path.append(r'c:\Users\as\OneDrive - ntpc.co.in\Desktop\Data Science\Kaggle Competitions')
from target_encoder import TargetEncoder

te = TargetEncoder(cols_to_encode=['code', 'sub_code', 'sub_category'],drop_original=True)
X_transformed = te.fit_transform(train_df, y=train_df['y_target'])
features = [col for col in X_transformed.columns if col not in ["id", "y_target", "weight"]]
target = "y_target"

percentile_80 = train_df['ts_index'].quantile(0.80)
X_train = X_transformed[X_transformed['ts_index'] <= percentile_80][features]
y_train = X_transformed[X_transformed['ts_index'] <= percentile_80][target]
w_train = X_transformed[X_transformed['ts_index'] <= percentile_80]['weight']

X_val = X_transformed[X_transformed['ts_index'] > percentile_80][features]
y_val = X_transformed[X_transformed['ts_index'] > percentile_80][target]
w_val = X_transformed[X_transformed['ts_index'] > percentile_80]['weight']

import lightgbm as lgb
import gc # Garbage collector for memory management

In [3]:
X_train_ts = X_train.drop(columns=['ts_index'])
X_val_ts = X_val.drop(columns=['ts_index'])

train_data = lgb.Dataset(X_train_ts, label=y_train, weight=w_train)
val_data = lgb.Dataset(X_val_ts, label=y_val, weight=w_val, reference=train_data)
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,     # Lower learning rate
    'num_leaves': 64,          # More complexity per tree
    'max_depth': -1,           # No limit on depth
    'feature_fraction': 0.7,   # Prevent overfitting 
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'lambda_l1': 1.0,          # L1 Regularization to handle noise
    'lambda_l2': 1.0,          # L2 Regularization
    'seed': 42,
    'verbosity': -1,
    'n_jobs': -1
}
print("Training LightGBM model...")

# We use early_stopping to prevent overfitting if validation score stops improving
model = lgb.train(
    params,
    train_data,
    num_boost_round=2000,           # Max number of trees
    valid_sets=[train_data, val_data], # Datasets to evaluate
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50), # Stop if no improvement for 50 rounds
        lgb.log_evaluation(period=50)           # Print progress every 50 rounds
    ]
)
print(f"Best iteration: {model.best_iteration}")
print(f"Best validation RMSE: {model.best_score['valid']['rmse']}")


# --- 4. Evaluate with Competition Metric ---
print("\nPredicting on validation set...")
y_pred_val = model.predict(X_val_ts, num_iteration=model.best_iteration)

score = weighted_rmse_score(y_val, y_pred_val, w_val)
print(f"\n>>> VALIDATION SCORE (Weighted RMSE): {score:.5f}")

del train_data, val_data
gc.collect()    

Training LightGBM model...
Training until validation scores don't improve for 50 rounds
[50]	train's rmse: 0.00198645	valid's rmse: 0.00270348
[100]	train's rmse: 0.00195038	valid's rmse: 0.00270362
Early stopping, best iteration is:
[50]	train's rmse: 0.00198645	valid's rmse: 0.00270348
Best iteration: 50
Best validation RMSE: 0.0027034840770585886

Predicting on validation set...

>>> VALIDATION SCORE (Weighted RMSE): 0.15213


24

In [4]:
feature_list = [col for col in X_train.columns if col not in ['id', 'y_target', 'weight', 'ts_index', 'code', 'sub_code', 'sub_category', 'horizon', 'TE_code_mean',
 'TE_sub_code_mean',
 'TE_sub_category_mean']]

In [24]:
group_cols = ['code', 'sub_code', 'sub_category', 'horizon']
df = train_df.sort_values(group_cols + ['ts_index'])
lag1_autocorr = (
    df.groupby(group_cols)['y_target']
      .apply(lambda x: x.autocorr(lag=1))
)

print(lag1_autocorr.mean())

c:\Users\as\miniconda3\envs\general\Lib\site-packages\numpy\lib\function_base.py:2889: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
c:\Users\as\miniconda3\envs\general\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
c:\Users\as\miniconda3\envs\general\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


0.550029609001809


In [ ]:
def safe_autocorr(x):
    if len(x) > 5:
        return x.autocorr(lag=1)
    return np.nan
feature_autocorr = {}
for feature in feature_list:
    autocorr = df.groupby(group_cols)[feature].apply(safe_autocorr)
    feature_autocorr[feature] = autocorr.mean()
    print(f"Autocorrelation for {feature}: {autocorr.mean()}")

In [30]:
# top 15 features by autocorrelation
top_features = sorted(feature_autocorr, key=feature_autocorr.get, reverse=True)[:20]
print("\nTop 15 features by autocorrelation:")
for feature in top_features:
    print(f"{feature}: {feature_autocorr[feature]:.4f}")



Top 15 features by autocorrelation:
feature_a: 0.9990
feature_bn: 0.9935
feature_j: 0.9875
feature_bh: 0.9861
feature_ah: 0.9844
feature_ch: 0.9814
feature_h: 0.9813
feature_bm: 0.9809
feature_bo: 0.9780
feature_bq: 0.9775
feature_be: 0.9688
feature_k: 0.9678
feature_bp: 0.9642
feature_ad: 0.9637
feature_v: 0.9634
feature_cb: 0.9628
feature_br: 0.9627
feature_ca: 0.9613
feature_r: 0.9590
feature_cc: 0.9560


In [32]:
# within group standard deviation of top features
for feature in top_features:
    std_dev = df.groupby(group_cols)[feature].std().mean()
    print(f"Average within-group std deviation and the corresponding autocorrelation for {feature}: {std_dev:.4f} ({feature_autocorr[feature]:.4f})")

Average within-group std deviation and the corresponding autocorrelation for feature_a: 43.8784 (0.9990)
Average within-group std deviation and the corresponding autocorrelation for feature_bn: 142.0652 (0.9935)
Average within-group std deviation and the corresponding autocorrelation for feature_j: 0.0002 (0.9875)
Average within-group std deviation and the corresponding autocorrelation for feature_bh: 200712.7081 (0.9861)
Average within-group std deviation and the corresponding autocorrelation for feature_ah: 1.2791 (0.9844)
Average within-group std deviation and the corresponding autocorrelation for feature_ch: 1.2752 (0.9814)
Average within-group std deviation and the corresponding autocorrelation for feature_h: 0.0006 (0.9813)
Average within-group std deviation and the corresponding autocorrelation for feature_bm: 138.3620 (0.9809)
Average within-group std deviation and the corresponding autocorrelation for feature_bo: 111.2773 (0.9780)
Average within-group std deviation and the cor

# Start lagging on imp features

In [6]:
# The base features you want to lag
features_to_lag = ['feature_a', 'feature_bn', 'feature_bh', 'feature_be', 'feature_bm']
group_cols = ['code', 'sub_code', 'sub_category', 'horizon']

# 1. Combine train and test to ensure continuous time series
# We add a marker to easily separate them later
train_df['is_test'] = 0
test_df['is_test'] = 1

combined_df = pd.concat([train_df, test_df], ignore_index=True)

# 2. Ensure everything is correctly sorted chronologically within each group
combined_df = combined_df.sort_values(group_cols + ['ts_index']).reset_index(drop=True)

# 3. Create the lag features
for feature in features_to_lag:
    combined_df[f"{feature}_lag1"] = combined_df.groupby(group_cols)[feature].shift(1)

# 4. Split them back apart
train_df = combined_df[combined_df['is_test'] == 0].drop(columns=['is_test'])
test_df = combined_df[combined_df['is_test'] == 1].drop(columns=['is_test'])

# Optional: clean up the large combined dataframe from memory
del combined_df
import gc
gc.collect()

for f in features_to_lag:
    train_df[f"{f}_lag1"].fillna(train_df[f], inplace=True)
    test_df[f"{f}_lag1"].fillna(test_df[f], inplace=True)


C:\Users\as\AppData\Local\Temp\ipykernel_34780\2317596028.py:29: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df[f"{f}_lag1"].fillna(train_df[f], inplace=True)
C:\Users\as\AppData\Local\Temp\ipykernel_34780\2317596028.py:30: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For 

In [7]:
# training using lagged features
te = TargetEncoder(cols_to_encode=['code', 'sub_code', 'sub_category'],drop_original=True)
X_transformed = te.fit_transform(train_df, y=train_df['y_target'])
features = [col for col in X_transformed.columns if col not in ["id", "y_target", "weight"]]
print(len(features))
target = "y_target"
percentile_80 = train_df['ts_index'].quantile(0.80)
X_train = X_transformed[X_transformed['ts_index'] <= percentile_80][features]
y_train = X_transformed[X_transformed['ts_index'] <= percentile_80][target]
w_train = X_transformed[X_transformed['ts_index'] <= percentile_80]['weight']

X_val = X_transformed[X_transformed['ts_index'] > percentile_80][features]
y_val = X_transformed[X_transformed['ts_index'] > percentile_80][target]
w_val = X_transformed[X_transformed['ts_index'] > percentile_80]['weight']


X_train_ts = X_train.drop(columns=['ts_index'])
X_val_ts = X_val.drop(columns=['ts_index'])

train_data = lgb.Dataset(X_train_ts, label=y_train, weight=w_train)
val_data = lgb.Dataset(X_val_ts, label=y_val, weight=w_val, reference=train_data)
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,     # Lower learning rate
    'num_leaves': 64,          # More complexity per tree
    'max_depth': -1,           # No limit on depth
    'feature_fraction': 0.7,   # Prevent overfitting 
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'lambda_l1': 1.0,          # L1 Regularization to handle noise
    'lambda_l2': 1.0,          # L2 Regularization
    'seed': 42,
    'verbosity': -1,
    'n_jobs': -1
}
print("Training LightGBM model...")

# We use early_stopping to prevent overfitting if validation score stops improving
model = lgb.train(
    params,
    train_data,
    num_boost_round=2000,           # Max number of trees
    valid_sets=[train_data, val_data], # Datasets to evaluate
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50), # Stop if no improvement for 50 rounds
        lgb.log_evaluation(period=50)           # Print progress every 50 rounds
    ]
)
print(f"Best iteration: {model.best_iteration}")
print(f"Best validation RMSE: {model.best_score['valid']['rmse']}")


# --- 4. Evaluate with Competition Metric ---
print("\nPredicting on validation set...")
y_pred_val = model.predict(X_val_ts, num_iteration=model.best_iteration)

score = weighted_rmse_score(y_val, y_pred_val, w_val)
print(f"\n>>> VALIDATION SCORE (Weighted RMSE): {score:.5f}")

del train_data, val_data
gc.collect()    





96
Training LightGBM model...
Training until validation scores don't improve for 50 rounds
[50]	train's rmse: 0.00198563	valid's rmse: 0.00270504
[100]	train's rmse: 0.00194866	valid's rmse: 0.00270475
Early stopping, best iteration is:
[70]	train's rmse: 0.00196897	valid's rmse: 0.00270359
Best iteration: 70
Best validation RMSE: 0.0027035885758224304

Predicting on validation set...

>>> VALIDATION SCORE (Weighted RMSE): 0.15188


556

In [8]:
X_test = te.transform(test_df)
X_test_ts = X_test[features].drop(columns=['ts_index'])
test_predictions = model.predict(X_test_ts, num_iteration=model.best_iteration)


c:\Users\as\OneDrive - ntpc.co.in\Desktop\Data Science\Kaggle Competitions\target_encoder.py:75: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_transformed[new_col_name].fillna(self.global_stats_[agg_func], inplace=True)
c:\Users\as\OneDrive - ntpc.co.in\Desktop\Data Science\Kaggle Competitions\target_encoder.py:75: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because

In [9]:
test_ids = test_df['id']
submission = pd.DataFrame({
    'id': test_ids,
    'prediction': test_predictions 
})
submission.to_csv('submission_lagged.csv', index=False)

In [10]:
submission

,id,prediction
0,10BAVIDU__07YQ9WA4__DPPUO5X2__1__4175,-0.358638
1,10BAVIDU__07YQ9WA4__DPPUO5X2__1__4176,-0.351284
2,10BAVIDU__07YQ9WA4__DPPUO5X2__1__4177,-0.198439
3,10BAVIDU__07YQ9WA4__DPPUO5X2__1__4178,-0.207606
4,10BAVIDU__07YQ9WA4__DPPUO5X2__1__4179,-0.207995
...,...,...
6771592,X9BZ68VQ__YKU5BVSL__V8BKY1IV__25__4372,2.107366
6771593,X9BZ68VQ__YKU5BVSL__V8BKY1IV__25__4373,2.095457
6771594,X9BZ68VQ__YKU5BVSL__V8BKY1IV__25__4374,2.095460
6771595,X9BZ68VQ__YKU5BVSL__V8BKY1IV__25__4375,2.095545
